In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from numpy import linalg as LA
from scipy.integrate import odeint
from multiprocessing import Pool
from math import cos, exp, pi, sin
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from numba import jit
import seaborn as sb
from matplotlib.lines import Line2D
mpl.use('Qt5Agg')
plt.ion()  

plt.rcParams['font.sans-serif'] = ['Times New Roman', 'SimHei']
plt.rcParams['mathtext.fontset'] = 'cm'       
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
def kuramoto(N, T, r1, r2, k1, k2, alpha, beta, initial):  
    def rhs(theta, t, N, r1, r2, k1, k2, alpha, beta,):
        dtheta_dt = np.zeros(N)

        local_order = np.zeros(N, dtype=np.complex64)
        phases = theta
        for ii in range(N):
            for j in range(-r2, r2 + 1):
                idx = (ii + j) % N
                local_order[ii] += np.exp(1j * phases[idx])
        
        local_order /= (2 * r2 + 1)
        epsilon = 1e-10
        threshold = 1e-12
        local_order[np.abs(local_order) < threshold] = epsilon

        O_alpha = np.abs(local_order) ** alpha 
        O_beta = np.abs(local_order) ** beta    

        for ii in range(N):
            for jj in range(-r1, r1 + 1):
                dtheta_dt[ii] += O_alpha[ii] * k1 * (np.sin(theta[(ii + jj) % N] - theta[ii])) / r1

        idx = list(range(-r2, 0)) + list(range(1, r2 + 1))
        for ii in range(N):
            for jj in idx:
                for kk in idx:
                    if jj != kk:
                        dtheta_dt[ii] += (
                            k2 * O_beta[ii]
                            * np.sin(
                                theta[(ii + kk) % N]
                                + theta[(ii + jj) % N]
                                - 2 * theta[ii]
                            )
                            / (r2 * (2.0 * r2 - 1))
                        )
        return dtheta_dt

    time = np.linspace(0, T, 1001)
    x = odeint(rhs, initial, time, args=(N, r1, r2, k1, k2, alpha, beta))
    return x

In [ ]:
def winding_number(xa, N, t):
    q = 0
    Delta = np.zeros(N)
    for ii in range(N):
        delta = xa[t, (ii + 1) % N] - xa[t, ii]
        if delta > np.pi:
            delta -= 2 * np.pi
        elif delta < -np.pi:
            delta += 2 * np.pi
        q += delta
        Delta[ii] = delta

    w_no = round(q / (2 * np.pi))
    Is_twisted_state = LA.norm(Delta - np.mean(Delta)) < 1e-1
    return w_no, Is_twisted_state


In [ ]:
def compute_Q(N, T, r1, r2, k1, k2, alpha, beta, Q):
    sync = np.linspace(0, 2 * np.pi * Q, N + 1)[:-1]
    if Q == 0:
        sync += np.pi
    p = 2 * np.pi * (np.random.rand(N) - 0.5)
    p -= np.mean(p)
    sync += 1e-5 * p  

    x = kuramoto(N, T, r1, r2, k1, k2, alpha, beta, sync)
    xa = x % (2 * np.pi)

    final_q, Is_twisted_state = winding_number(xa, N, -1)
    return 1 if final_q == Q or final_q == N - Q or final_q == Q - N else 0


In [ ]:
@jit
def lambda2(N, K1, K2, q, r, beta):

    if q == 0:
        O_beta = 1
    else:
        a = 2 * np.pi * q / N
        numerator = np.sin((r + 1/2) * a)
        denominator = (2 * r + 1) * np.sin(a / 2)
        Oj_abs = np.abs(numerator / denominator)

    O_beta = Oj_abs ** beta 

    a2 = K1 * cos(2 * np.pi * q / N) / (2 * r)
    a3 = K1 * cos(4 * np.pi * q / N) / (2 * r)
    a1 = -2 * (a2 + a3)

    b2 = K2 * O_beta * (1 + cos(2 * np.pi * q / N) + cos(6 * np.pi * q / N)) / (r * (2 * r - 1))
    b3 = K2 * O_beta *  (1 + cos(2 * np.pi * q / N) + cos(6 * np.pi * q / N)) / (r * (2 * r - 1))
    b1 = -2 * (b2 + b3)

    A1 = a1 + b1
    A2 = a2 + b2
    A3 = a3 + b3

    Lambda = np.zeros(N, dtype=np.complex128)
    for i in range(N):    
        Lambda[i] = (
            A1
            + A2 * np.exp(2j * np.pi * i / N)
            + A3 * np.exp(2j * np.pi * i * 2 / N)
            + A2 * np.exp(2j * np.pi * i * (N - 1) / N)
            + A3 * np.exp(2j * np.pi * i * (N - 2) / N)
        )

    Lambda = np.real(Lambda)
    lambda_2 = np.sort(Lambda[1:])[-1]

    return lambda_2


In [ ]:
N = 83
T = 200  
r1 = 2
r2 = 2
k1 = 1
k2 = 2
alpha = 0
beta = 0.5

Q_values = np.arange(N)
is_values = [compute_Q(N, T, r1, r2, k1, k2, alpha, beta, Q) for Q in Q_values]  
data = np.column_stack((Q_values, is_values))
# np.savetxt('stability_data_0.5_k2_2.csv', data, delimiter=',', header='Q_values,is_values', comments='')

In [ ]:
plt.close('all')

N = 83
r = 2
k1 = 1
beta_list = [0.5, 2]
k2_list = [1, 4]
colors = [(0.0, 0.45, 0.74), (0.8, 0.0, 0.0)]
labels_beta = [r"$\beta=0.5$", r"$\beta=2$"]
labels_k2 = [r"$k_2=1$", r"$k_2=4$"]

zz = 20
zzz = 22

def transform_q(q_vals, N=83):
    q_new = q_vals.copy()
    q_new[q_vals >= 42] = q_new[q_vals >= 42] - N
    return q_new

def sort_by_q(q, y):
    idx = np.argsort(q)
    return q[idx], y[idx]

data_b0 = np.genfromtxt('stability_data0_0.5_k2_2.csv', delimiter=',', skip_header=1)
data_b1 = np.genfromtxt('stability_data0_2_k2_2.csv', delimiter=',', skip_header=1)
Q_b0, is_b0 = sort_by_q(transform_q(data_b0[:, 0]), data_b0[:, 1])
Q_b1, is_b1 = sort_by_q(transform_q(data_b1[:, 0]), data_b1[:, 1])

data_k2_05 = np.genfromtxt('stability_data0_1_k2_1.csv', delimiter=',', skip_header=1)
data_k2_2 = np.genfromtxt('stability_data0_1_k2_4.csv', delimiter=',', skip_header=1)
Q_k2_05, is_k2_05 = sort_by_q(transform_q(data_k2_05[:, 0]), data_k2_05[:, 1])
Q_k2_2, is_k2_2 = sort_by_q(transform_q(data_k2_2[:, 0]), data_k2_2[:, 1])

def compute_lambda_dict(param_list, param_type='beta'):
    result = {}
    q_raw = np.arange(N)
    q_transformed = transform_q(q_raw)
    q_sorted, _ = sort_by_q(q_transformed, np.zeros_like(q_transformed))
    for param in param_list:
        vals = np.zeros(N, dtype=np.complex128)
        for q in range(N):
            if param_type == 'beta':
                vals[q] = lambda2(N, k1, 2, q, r, param)
            else:
                vals[q] = lambda2(N, k1, param, q, r, 1)
        result[param] = sort_by_q(q_transformed, vals.real)[1]
    return q_sorted, result

q_sorted, lambda_beta = compute_lambda_dict(beta_list, 'beta')
_, lambda_k2 = compute_lambda_dict(k2_list, 'k2')

fig, axes = plt.subplots(3, 2, figsize=(8, 7), sharex='col',
                         gridspec_kw={'hspace': 0.4, 'wspace': 0.3,
                                      'height_ratios': [1, 1, 0.5]})

axes[0, 0].plot(Q_b0, is_b0, '-', color=colors[0])
axes[0, 0].plot(Q_b1, is_b1, '--', color=colors[1])
axes[0, 0].set_yticks([0, 1])
axes[0, 0].tick_params(labelsize=zz)
axes[0, 0].set_ylabel("Flag", fontsize=zzz)
axes[0, 0].text(0.02, 1.05, "(a)", transform=axes[0, 0].transAxes, fontsize=zzz)

for i, beta in enumerate(beta_list):
    axes[1, 0].plot(q_sorted, lambda_beta[beta], '--' if i == 1 else '-', color=colors[i], label=labels_beta[i], linewidth=1.5)
axes[1, 0].axhline(0, ls="--", color="gray", alpha=0.6)
axes[1, 0].tick_params(labelsize=zz)
axes[1, 0].set_ylabel(r"$\lambda_\mathrm{max}$", fontsize=zzz)
axes[1, 0].legend(fontsize=zz - 2, frameon=False, loc='upper center')
axes[1, 0].text(0.02, 1.05, "(c)", transform=axes[1, 0].transAxes, fontsize=zzz)
axes[1, 0].set_yticks([0, 1, 2]) 
axes[1, 0].legend(fontsize=zz - 2, frameon=False,
                  loc='upper center', bbox_to_anchor=(0.5, 1.15))  

for i, beta in enumerate(beta_list):
    axes[2, 0].plot(q_sorted, lambda_beta[beta], '--' if i == 1 else '-', color=colors[i], linewidth=1.2)
axes[2, 0].set_ylim(-0.06, 0.001)
axes[2, 0].axhline(0, ls="--", color="gray", alpha=0.6)
axes[2, 0].tick_params(labelsize=zz)
axes[2, 0].set_xlabel(r"$q$", fontsize=zzz)
axes[2, 0].set_ylabel(r"$\lambda_\mathrm{max}$", fontsize=zzz)
axes[2, 0].text(0.02, 1.05, "(e)", transform=axes[2, 0].transAxes, fontsize=zzz)

axes[0, 1].plot(Q_k2_05, is_k2_05, '-', color=colors[0])
axes[0, 1].plot(Q_k2_2, is_k2_2, '--', color=colors[1])
axes[0, 1].set_yticks([0, 1])
axes[0, 1].tick_params(labelsize=zz)
axes[0, 1].text(0.02, 1.05, "(b)", transform=axes[0, 1].transAxes, fontsize=zzz)

for i, k2 in enumerate(k2_list):
    axes[1, 1].plot(q_sorted, lambda_k2[k2], '--' if i == 1 else '-', color=colors[i], label=labels_k2[i], linewidth=1.5)
axes[1, 1].axhline(0, ls="--", color="gray", alpha=0.6)
axes[1, 1].tick_params(labelsize=zz)
axes[1, 1].legend(fontsize=zz - 2, frameon=False, loc='upper center')
axes[1, 1].text(0.02, 1.05, "(d)", transform=axes[1, 1].transAxes, fontsize=zzz)
axes[1, 1].set_yticks([0, 1, 2])  
axes[1, 1].legend(fontsize=zz - 2, frameon=False,
                  loc='upper center', bbox_to_anchor=(0.5, 1.15))  

for i, k2 in enumerate(k2_list):
    axes[2, 1].plot(q_sorted, lambda_k2[k2], '--' if i == 1 else '-', color=colors[i], linewidth=1.2)
axes[2, 1].set_ylim(-0.06, 0.001)
axes[2, 1].axhline(0, ls="--", color="gray", alpha=0.6)
axes[2, 1].tick_params(labelsize=zz)
axes[2, 1].set_xlabel(r"$q$", fontsize=zzz)
axes[2, 1].text(0.02, 1.05, "(f)", transform=axes[2, 1].transAxes, fontsize=zzz)

sb.despine()
plt.tight_layout()
# plt.savefig("fig2.eps", format="eps",dpi=300, bbox_inches="tight", pad_inches=0)
plt.show()
